## **Feature Engineering**

Learning outcomes

* Define feature engineering and explain its critical role in predictive performance ("Garbage in, garbage out").

* Categorize variable types (nominal, ordinal, continuous, datetime) and select corresponding transformation techniques.

* Implement categorical encoding techniques (One-Hot, Ordinal, Target, Frequency) using Pandas and Scikit-Learn.

* Apply feature scaling methods (Standard, Min-Max, Robust) and mathematical transformations appropriately.

* Construct complex composite, temporal, and interaction features.

* Structure data preprocessing within Scikit-Learn Pipeline objects to strictly prevent data leakage.

Feature engineering is the process of using domain knowledge and statistical transformations to extract, convert, and construct features (variables) from raw data into representations that maximize machine learning model performance.

Issues with Raw Data

> Mathematical Constraints: Most algorithms (e.g., Linear Regression, SVM, Neural Networks) operate on mathematical vectors and matrices; they cannot process raw string categories or timestamps directly.

> Non-Linear Relationships: Linear algorithms struggle with complex interaction effects or non-linear curves unless explicitly feature-engineered.

> Magnitude Bias: Features measured on different scales (e.g., Age in years [0–100] vs. Income in USD [0–500,000]) cause distance-based algorithms to weigh larger scales disproportionately.

**Categorical Data Encoding**

Categorical features store discrete labels. Models require numerical conversions, but improper encoding can introduce false assumptions (such as artificial ordering).

**Ordinal Encoding:**
* This is used for ordinal data types (e.g. Ticket Booking class, Semester of a student, etc)
* The procedure maps ordered categories to sequential integers (0,1,2...)
* Preserves the inherent ranking
* However, some classes may not be linear. Implies equal distance between ranks if misapplied to nominal data.

**One-Hot Encoding (OHE)**

* Nominal variables (low cardinality) (e.g. Mango, Apple)
* Creates K binary indicator columns
* Does not work for large dimensions


**Target (mean) Encoding**
* used for high cardinality (all items in supermarket)
* Replaces category with target mean  $E[y \mid X = c_i]$.
* captures target signal
* Carries risk of overfitting due to presence of large number of variables

**Frequency/Count Encoding**

* Nominal (high cardinality), replaces category with frequency or count of occurrences,
* simple
* categories with identical counts end up with identical value

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Sample Dataset
data = {
    'Education': ['High School', 'Bachelors', 'Masters', 'PhD', 'Bachelors'],
    'City': ['NYC', 'London', 'Paris', 'NYC', 'London'],
    'Target': [0, 1, 1, 0, 1]
}
df = pd.DataFrame(data)

Ordinal Encoding
For variables where order carries explicit semantic meaning (`High School < Bachelors < Masters < PhD`).

In [ ]:
# Specify explicit order to avoid arbitrary numerical assignment
education_order = [['High School', 'Bachelors', 'Masters', 'PhD']]
ord_enc = OrdinalEncoder(categories=education_order)

df['Education_Encoded'] = ord_enc.fit_transform(df[['Education']])
print(df[['Education', 'Education_Encoded']])

     Education  Education_Encoded
0  High School                0.0
1    Bachelors                1.0
2      Masters                2.0
3          PhD                3.0
4    Bachelors                1.0


One-Hot Encoding (OHE)
For nominal variables with low cardinality (e.g., short lists of cities or categories).

In [ ]:
# 1. Pandas approach
df_ohe = pd.get_dummies(df, columns=['City'], drop_first=True, dtype=int)
# drop_first=True prevents the "Dummy Variable Trap" (Multicollinearity)

# 2. Scikit-Learn approach (preferred inside ML pipelines)
ohe = OneHotEncoder(sparse_output=False, drop='first')
city_encoded = ohe.fit_transform(df[['City']])
city_df = pd.DataFrame(city_encoded, columns=ohe.get_feature_names_out(['City']))

Target (Mean) Encoding with Smoothing

For nominal variables with high cardinality (e.g., zip codes, product IDs). Target encoding replaces category $c_i$ with the smoothed mean target value $S_i$:

$$S_i = \lambda(n_i) \cdot \bar{y}_i + (1 - \lambda(n_i)) \cdot \bar{y}_{\text{global}}$$

where $\lambda(n_i) = \frac{n_i}{n_i + m}$ ($n_i$ is category count, $m$ is smoothing weight, and $\bar{y}_{\text{global}}$ is the overall dataset target mean).

In [ ]:
# Calculating smoothed target encoding manually
global_mean = df['Target'].mean()
m_weight = 10  # Smoothing parameter

stats = df.groupby('City')['Target'].agg(['count', 'mean'])
smooth_encoding = (stats['count'] * stats['mean'] + m_weight * global_mean) / (stats['count'] + m_weight)

df['City_Target_Encoded'] = df['City'].map(smooth_encoding)
print(df[['City', 'City_Target_Encoded']])

     City  City_Target_Encoded
0     NYC             0.500000
1  London             0.666667
2   Paris             0.636364
3     NYC             0.500000
4  London             0.666667


Feature Scaling & Transformations

Models sensitive to scale include distance-based algorithms (K-NN, SVM, K-Means) and gradient descent optimization algorithms (Linear/Logistic Regression, Neural Networks). Tree-based models (Random Forest, XGBoost) are invariant to monotonic transformations.

**Standard Scaling (Z-score Normalization)**

Rescales data to zero mean ($\mu = 0$) and unit variance ($\sigma = 1$).
$$x' = \frac{x - \mu}{\sigma}$$

When to use: Features following a Gaussian/Normal distribution; algorithms using distance metrics or gradient descent.

**Min-Max Scaling (Normalization)**

Rescales data strictly within a fixed range (typically $[0, 1]$).
$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

When to use: Bounded features, neural network inputs, or algorithms requiring non-negative values.

**Robust Scaling**

Rescales data using the median ($Q_2$) and Interquartile Range ($\text{IQR} = Q_3 - Q_1$).

$$x' = \frac{x - \text{median}}{\text{IQR}}$$

When to use: Datasets containing significant outliers that would distort $\mu$ and $\sigma$ in Standard Scaling.

Power Transformations (Log, Box-Cox)

Compresses heavy right-skewed distributions toward normality.

$$x' = \ln(x + 1) \quad (\text{Log for values } x \ge 0)$$

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Sample Matrix with an Outlier
X = np.array([[10.0, 1000.0],
              [20.0, 2000.0],
              [15.0, 1500.0],
              [100.0, 50000.0]]) # Outlier in 2nd column

# Implement Scalers
X_std = StandardScaler().fit_transform(X)
X_minmax = MinMaxScaler().fit_transform(X)
X_robust = RobustScaler().fit_transform(X)

print("Robust Scaled (Resistant to Extreme Values):\n", X_robust)

Robust Scaled (Resistant to Extreme Values):
 [[-0.28571429 -0.05940594]
 [ 0.0952381   0.01980198]
 [-0.0952381  -0.01980198]
 [ 3.14285714  3.82178218]]


**Derived & Domain-Specific Feature Creation**

A. Interaction FeaturesCombining two or more numeric features to capture non-linear relationships.

Ratio Features:

$\text{Debt-to-Income Ratio} = \frac{\text{Total Debt}}{\text{Annual Income}}$

Product Features:

$\text{Total Fare} = \text{Ticket Price} \times \text{Group Size}$

**Binning / Discretization**
Converting continuous features into discrete bins to reduce sensitivity to small noise or handle non-monotonic effects.

In [ ]:
df_age = pd.DataFrame({'Age': [18, 22, 25, 35, 42, 58, 65, 71, 80]})

# 1. Equal-Width Binning (Divides total range into equal interval lengths)
df_age['Age_Width_Bins'] = pd.cut(df_age['Age'], bins=3, labels=['Young', 'Middle', 'Senior'])

# 2. Equal-Frequency Binning (Quantile-based: equal number of samples per bin)
df_age['Age_Freq_Bins'] = pd.qcut(df_age['Age'], q=3, labels=['Low', 'Mid', 'High'])

print(df_age)

   Age Age_Width_Bins Age_Freq_Bins
0   18          Young           Low
1   22          Young           Low
2   25          Young           Low
3   35          Young           Mid
4   42         Middle           Mid
5   58         Middle           Mid
6   65         Senior          High
7   71         Senior          High
8   80         Senior          High


**Datetime Feature Extraction**
Extracting underlying cyclic and contextual signals from timestamp fields.

In [ ]:
df_time = pd.DataFrame({'Timestamp': pd.date_range(start='2026-01-01', periods=5, freq='D')})

df_time['DayOfWeek'] = df_time['Timestamp'].dt.day_name()
df_time['IsWeekend'] = (df_time['Timestamp'].dt.dayofweek >= 5).astype(int)
df_time['DayOfYear'] = df_time['Timestamp'].dt.dayofyear
print(df_time)

   Timestamp DayOfWeek  IsWeekend  DayOfYear
0 2026-01-01  Thursday          0          1
1 2026-01-02    Friday          0          2
2 2026-01-03  Saturday          1          3
3 2026-01-04    Sunday          1          4
4 2026-01-05    Monday          0          5


**Preventing Data Leakage with Pipelines**

Data Leakage occurs when information from outside the training dataset (such as mean, variance, or target statistics from test or validation sets) leaks into model training.

*Rules of Preprocessing Hygiene:*

1. Split First: Always execute train_test_split prior to calculating scaling metrics ($\mu, \sigma$), missing value imputation values, or target encodings.

2. Fit on Train, Transform on Test: Execute .fit() or .fit_transform() exclusively on the training partition (X_train), and only .transform() on testing partitions (X_test).

3. Use Scikit-Learn Pipelines: Encapsulate transformers and estimators to streamline execution.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Define column groups
num_cols = ['age', 'fare']
cat_cols = ['embarked', 'sex']

# Numerical transformation pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical transformation pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# Full model pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Correct execution flow:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
# full_pipeline.fit(X_train, y_train)           # Computes params from X_train ONLY
# predictions = full_pipeline.predict(X_test)   # Applies learned params to X_test

***Exercise:***
Imagine an e-commerce credit card transaction dataset containing raw features:

['user_id', 'transaction_time', 'purchase_amount', 'user_country', 'device_type'].



> Working in pairs, design 5 engineered features optimized for predicting fraudulent transactions.

What temporal features could capture suspicious velocity? (e.g., number of transactions in the past 10 minutes, time since last password reset).

How would you encode user_country if it contains 195 distinct values? (Compare Target Encoding vs. Frequency Encoding vs. One-Hot Encoding top 10 countries + "Other").

Why is scaling necessary if training a Logistic Regression model, but optional for XGBoost?